In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import re

# preprocessing mutation.tsv

In [2]:
# import re

input_file = '../data/raw/mutations.tsv'
output_file = '../data/raw/mutations_clean.tsv'

pattern = re.compile(r'^EBI-\d+')

clean_lines = []
current_line = ''

with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        if pattern.match(line):
            if current_line:
                clean_lines.append(current_line.strip() + '\n')
            current_line = line.strip()
        else:

            current_line += line.strip()

if current_line:
    clean_lines.append(current_line.strip() + '\n')

with open(output_file, 'w', encoding='utf-8') as f:
    f.writelines(clean_lines)


In [3]:
fileName = r'../data/raw/mutations_clean.tsv'
df = pd.read_csv(fileName, sep='\t', keep_default_na=False)
print(df.shape)
df.head()

(58250, 15)


,#Feature AC,Feature short label,Feature range(s),Original sequence,Resulting sequence,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC
0,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],83-83,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
1,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],87-87,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
2,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],91-91,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
3,EBI-6925687,p.Cys169Ser,169-169,C,S,mutation(MI:0118),,uniprotkb:P0A6H1,clpX,,83333 - Escherichia coli (strain K12),"uniprotkb:P0A6H1(protein(MI:0326), 83333 - Esc...",23622246,Supp Fig. 2A,EBI-6925660
4,EBI-6898360,p.Phe508del,508-508,F,.,mutation(MI:0118),,uniprotkb:P13569,CFTR,,9606 - Homo sapiens,"uniprotkb:P13569(protein(MI:0326), 9606 - Homo...",22038833,"1B, 4B",EBI-6898336


In [4]:
df[(df['Feature annotation'].str.contains('high-throughput'))].shape

(11028, 15)

In [5]:
# drop high-throught
df = df[~(df['Feature annotation'].str.contains('high-throughput'))]
df.shape

(47222, 15)

## drop entries with >2 participants, and drop entries that the number of partner don't match the number of uniprotAC. (to filter binary protein-protein interaction)

In [6]:
import re
p = re.compile(r'uniprotkb:(.*?)[(]', re.S)
partner = []
n_partner = []
count = 0
for i in df['Interaction participants']:
    tmp = re.findall(p, i)
#     num = re.findall(p2, i)
    num = i.count(';') + 1
    partner.append(tmp)
    n_partner.append(num)
df['partners'] = partner
df['n_partner'] = n_partner

df = df[df['n_partner'] < 3]
print('after delete items with more than 2 partners {}'.format(df.shape))
df = df[df['partners'].apply(lambda x: len(x)) == df['n_partner']]
print('after delete items with not identical number of partners and n_partner {}'.format(df.shape))

after delete items with more than 2 partners (43126, 17)
after delete items with not identical number of partners and n_partner (35267, 17)


## drop entries with same interactionAC but different affected protein AC (drop same interaction with multiple mutations)

In [7]:
df1 = df[df.duplicated(['Affected protein AC', 'Interaction AC'], keep=False)] # choose items with same interactAC-aff pro AC pair
df2 = df.drop_duplicates(['Interaction AC'], keep=False) # choose items with only one time interactionAC
df = pd.concat([df1, df2])
print(df.shape)

(34712, 17)


## drop entries without uniprotAC

In [8]:
df = df[df['Affected protein AC'].str.contains('uniprotkb:', na=False)]
print(df.shape)

(34696, 17)


## simplify uniprotkb label

In [9]:
df['Affected protein AC'] = df['Affected protein AC'].str.replace('uniprotkb:', '')
df.head()

,#Feature AC,Feature short label,Feature range(s),Original sequence,Resulting sequence,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner
0,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],83-83,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
1,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],87-87,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
2,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],91-91,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
5,EBI-6925862,p.Cys169Ser,169-169,C,S,mutation(MI:0118),,P0A6H1,clpX,,83333 - Escherichia coli (strain K12),"uniprotkb:P0A6H1(protein(MI:0326), 83333 - Esc...",23622246,Supp Fig. 2C,EBI-6925855,"[P0A6H1, P0A6H1]",2
13,EBI-8875481,p.Ile204Tyr,204-204,I,Y,mutation(MI:0118),,Q8TE30,q8te30_human,,9606 - Homo sapiens,"uniprotkb:Q8TE30(protein(MI:0326), 9606 - Homo...",24267889,"Fig. 6F, Supp. Fig. 7H",EBI-8875425,"[Q8TE30, Q9UN81]",2


## delete 'mutation' feature type

In [10]:
df = df[~df['Feature type'].isin(['mutation(MI:0118)'])]
print(df.shape)

(31792, 17)


## delete non- regular acid items with same featureAC

In [11]:
f_ = df[df['Resulting sequence'].str.contains('B|J|O|Z', na=False)]['#Feature AC'].tolist()
df = df[~df['#Feature AC'].isin(f_)]
print(df.shape)
print(f_)

(31791, 17)
['EBI-8291032']


## delete 'PRO_' uniprotAC in table

In [12]:
df = df[~df['Affected protein AC'].str.contains('PRO_')]
df.shape

(31488, 17)

## get all sequence from uniprot (prepare for uniprot fasta retrieve https://www.uniprot.org/uploadlists/)

In [13]:
def flatlist(acList):
    return [item for sublist in acList for item in sublist]

ac1 = set(flatlist(df['partners'].values.tolist()))
ac2 = set(df['Affected protein AC'].values.tolist())
acAll = ac1 | ac2
with open('../data/middlefile/acAll.txt', 'w') as f:
    for x in acAll:
        f.write(x + '\n')


In [14]:
p = re.compile('PRO_')
acAll = [x for x in acAll if not p.findall(x)]
len(acAll)

7574

In [15]:
ac = []
info = []
seq = []
seqline = ''
initFlag = True
fastaFile = '../data/raw/allAC.fasta' # from uniprot website mapping, download with canonical and isoform
with open(fastaFile, 'r') as f:
    for line in f:
        line = line.strip()
        if '>' in line:
            res = re.findall(r'\|([^"]+)\|', line)[0]
            ac.append(res)
            info.append(line)
            if initFlag:
                initFlag = False
            else:
                seq.append(seqline)
                seqline = ''
        else:
            seqline += line
    seq.append(seqline)
fastaTable = pd.DataFrame({'ac': ac, 'info': info, 'seq': seq})

In [16]:
import pandas as pd

dup_counts = fastaTable['ac'].value_counts()
duplicates_ac = dup_counts[dup_counts > 1].index.tolist()

print(len(duplicates_ac))

for ac in duplicates_ac:
    seqs = fastaTable.loc[fastaTable['ac'] == ac, 'seq']
    if seqs.nunique() > 1:
        print(ac)

351


## select valid uniprotAC to make following selection

In [17]:
validAC1 = fastaTable[fastaTable['ac'].isin(acAll)]

validAC2 = fastaTable[~fastaTable['ac'].isin(acAll)]
acAll_series = pd.Series(list(acAll))
validAC2 = validAC2[validAC2['ac'].isin(acAll_series.str.split('-', expand=True)[0])]
validAC = pd.concat([validAC1, validAC2])
print(validAC1.shape)
print(validAC2.shape)
print(validAC.shape)

(7198, 3)
(354, 3)
(7552, 3)


## make the 'affected protein AC' - 'uniprotAC' dict. Some have 'multiple key' -> 'single value' relationship eg: apac['P19838-1'] = 'P19838', apac['P19838'] = 'P19838'

In [18]:
apacKey = []
acValue = []
for ac in acAll:
    if ac in validAC['ac'].values:
        apacKey.append(ac)
        acValue.append(ac)
    elif ac.split('-')[0] in validAC['ac'].values:
        apacKey.append(ac)
        acValue.append(ac.split('-')[0])
apac2ac = dict(zip(apacKey, acValue))


### transform all isoform AC in table into real uniprotAC(canonical with no isoform '-'), eg: O43889-2 ->O43889, O43889-3 -> O43889-3

In [19]:
df = df[df['Affected protein AC'].isin(apac2ac.keys())]

In [20]:
df = df[df['partners'].apply(lambda x: set(x) < set(list(apac2ac.keys())))]

## make interaction with multi position mutations into one 

In [21]:
# pos = df['Feature range(s)'].str.split('-', expand=True)
# df['start'] = pos[0]
# df['end'] = pos[1]

comCol = df.columns.tolist()
comCol.remove('Feature range(s)')
comCol.remove('Original sequence')
comCol.remove('Resulting sequence')

df_1 = df.groupby('#Feature AC')[['Feature range(s)','Original sequence', 'Resulting sequence']].agg(list)
df_2 = df[comCol].drop_duplicates('#Feature AC', keep='first')
df = pd.merge(df_1, df_2, on = '#Feature AC')
df.reset_index(drop=True, inplace=True)
df.shape

(26816, 17)

In [22]:
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039307,"[Q03694, P28795]",2
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039491,"[Q03694, P28795]",2
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2a f2b,EBI-10039532,"[P28795, Q03694]",2
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2c,EBI-10039697,"[P28795, Q03694]",2
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2d,EBI-10039716,"[P28795, Q03694]",2


## make mutprotein seq and participants seq

In [23]:
validAC_index = validAC.copy()
validAC_index = validAC_index.drop_duplicates(subset='ac', keep='first') #new_process
validAC_index = validAC_index.set_index('ac')

In [24]:
validAC_index.head()

,info,seq
ac,,
P32639,>sp|P32639|BRR2_YEAST Pre-mRNA-splicing helica...,MTEHETKDKAKKIREIYRYDEMSNKVLKVDKRFMNTSQNPQRDAEI...
P40424,>sp|P40424|PBX1_HUMAN Pre-B-cell leukemia tran...,MDEQPRLMHSHAGVGMAGHPGLSQHLQDGAGGTEGEGGRKQDIGDI...
P0C0L4,>sp|P0C0L4|CO4A_HUMAN Complement C4-A OS=Homo ...,MRLLWGLIWASSFFTLSLQKPRLLLFSPSVVHLGVPLSVGVQLQDV...
Q9BXC9,>sp|Q9BXC9|BBS2_HUMAN BBSome complex member BB...,MLLPVFTLKLRHKISPRMVAIGRYDGTHPCLAAATQTGKVFIHNPH...
P27884,>sp|P27884|CAC1A_RABIT Voltage-dependent P/Q-t...,MARFGDEMPARYGGGGAGAAAGVVVGAAGGRGAGGSRQGGQPGAQR...


In [25]:
mutAC = [apac2ac[x] for x in df['Affected protein AC']]
mut0 = []
for i in mutAC:
    mut0.append(validAC_index.loc[i, 'seq'])

df['mut0'] = mut0

In [26]:
par = []
parAC = []

for i in df.index:
    sameFlag = False
    if len(df.loc[i, 'partners']) > 1:
        for j in df.loc[i, 'partners']:
            if j != df.loc[i, 'Affected protein AC']:
                par.append(validAC_index.loc[apac2ac[j], 'seq'])
                parAC.append(apac2ac[j])
            elif sameFlag:
                par.append(validAC_index.loc[apac2ac[j], 'seq'])
                parAC.append(apac2ac[j])
            else:
                sameFlag = True
    elif len(df.loc[i, 'partners']) == 1:
        par.append(validAC_index.loc[apac2ac[df.loc[i, 'partners'][0]], 'seq'])
        parAC.append(apac2ac[df.loc[i, 'partners'][0]])

df['par0'] = par
df['parAC'] = parAC
print(len(par))
print(df.shape)

26816
(26816, 20)


In [27]:
# df.loc[i, 'mut0']

df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner,mut0,par0,parAC
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039307,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039491,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2a f2b,EBI-10039532,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2c,EBI-10039697,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2d,EBI-10039716,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694


In [28]:
df['Resulting sequence'] = df['Resulting sequence'].apply(
    lambda seqs: [s.replace('.', '') for s in seqs]
)
df.head()
    

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner,mut0,par0,parAC
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039307,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039491,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2a f2b,EBI-10039532,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2c,EBI-10039697,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2d,EBI-10039716,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694


In [29]:
mach = []
unmach = []
for i in range(len(df)):
    correct = True
    sample = df.iloc[i]
    for j in range(len(sample['Feature range(s)'])):
        positions = sample['Feature range(s)'][j].split('-')
        start = int(positions[0]) - 1
        end = int(positions[1])
        if sample['mut0'][start : end] != sample['Original sequence'][j]:
            unmach.append(sample)
            correct = False
            break
    if correct:
        mach.append(sample)
        

In [30]:
print(len(mach), len(unmach))

26086 730


In [31]:
df = pd.DataFrame(mach)
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner,mut0,par0,parAC
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039307,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039491,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2a f2b,EBI-10039532,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2c,EBI-10039697,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2d,EBI-10039716,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694


In [32]:
def sort_by_feature_ranges(row):

    combined = list(zip(row['Feature range(s)'], 
                        row['Original sequence'], 
                        row['Resulting sequence']))
    

    combined.sort(key=lambda x: int(x[0].split('-')[0]))
    

    feature_ranges, original_seq, resulting_seq = zip(*combined)
    
    row['Feature range(s)'] = list(feature_ranges)
    row['Original sequence'] = list(original_seq)
    row['Resulting sequence'] = list(resulting_seq)
    return row

df = df.apply(sort_by_feature_ranges, axis=1)

In [33]:
# df['positions_mut0'] = df['positions_mut0'].apply(
#     lambda x: sorted(x, key=lambda s: int(s.split('-')[0])) if isinstance(x, list) else x
# )

In [34]:
from tqdm import tqdm
all_positions_mut0 = []
all_positions_mut1 = []
all_mut1 = []
for i in tqdm(range(len(df))):
    sample = df.iloc[i]
    parts = []
    end_before = 0
    accumulate = 0
    positions_mut0 = []
    positions_mut1 = []
    for j in range(len(sample['Feature range(s)'])):
        positions = sample['Feature range(s)'][j].split('-')
        start = int(positions[0]) - 1
        parts.append(sample['mut0'][end_before: start])
        end_before = int(positions[1])
        positions_mut0.append(str(start) + '-'  + str(end_before))
        positions_mut1.append(str(start + accumulate) + '-' + str(end_before + accumulate + len(sample['Resulting sequence'][j]) - len(sample['Original sequence'][j])))
        accumulate = accumulate + len(sample['Resulting sequence'][j]) - len(sample['Original sequence'][j])
    parts.append(sample['mut0'][end_before : ])
    mut1 = ''
    mut0 = ''
    for j in range(len(sample['Feature range(s)'])):
        mut1 = mut1 + parts[j] + sample['Resulting sequence'][j]
        mut0 = mut0 + parts[j] + sample['Original sequence'][j]
    mut1 = mut1 + parts[-1]
    mut0 = mut0 + parts[-1]
    if mut0 != sample['mut0']:
        print('err', i)
        break
    all_mut1.append(mut1)
    all_positions_mut0.append(positions_mut0)
    all_positions_mut1.append(positions_mut1)
    # if sample['#Feature AC'] == 'EBI-10767274':
    #     break

df['mut1'] = all_mut1
df['positions_mut0'] = all_positions_mut0
df['positions_mut1'] = all_positions_mut1

100%|██████████| 26086/26086 [00:10<00:00, 2566.54it/s]


In [35]:
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,...,Figure legend,Interaction AC,partners,n_partner,mut0,par0,parAC,mut1,positions_mut0,positions_mut1
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,f1c,EBI-10039307,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,[80-81],[80-81]
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,...,f1c,EBI-10039491,"[Q03694, P28795]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,[187-188],[187-188]
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,f2a f2b,EBI-10039532,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,[80-81],[80-81]
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,f2c,EBI-10039697,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,[80-81],[80-81]
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,f2d,EBI-10039716,"[P28795, Q03694]",2,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,Q03694,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,[80-81],[80-81]


In [36]:
df['label'] = 2
df.loc[df['Feature type'].str.contains('disrupting'), 'label'] = 0
df.loc[df['Feature type'].str.contains('decreasing'), 'label'] = 1
df.loc[df['Feature type'].str.contains('increasing'), 'label'] = 3
df.loc[df['Feature type'].str.contains('causing'), 'label'] = 4

In [37]:
df.to_pickle('../data/processed/mutations_correct.dataset')